### Install and Import Libraries

In [1]:
!pip install pandas
import pandas as pd

### Load and Inspect Raw Data

In [2]:
# Load raw dataset from file path
df = pd.read_csv(r"Dataset\raw_dataset\Sales Dataset.csv")
# Preview first few rows to understand structure 
df.head()
# check data types, null values and overall structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Order ID      1194 non-null   object
 1   Amount        1194 non-null   int64 
 2   Profit        1194 non-null   int64 
 3   Quantity      1194 non-null   int64 
 4   Category      1194 non-null   object
 5   Sub-Category  1194 non-null   object
 6   PaymentMode   1194 non-null   object
 7   Order Date    1194 non-null   object
 8   CustomerName  1194 non-null   object
 9   State         1194 non-null   object
 10  City          1194 non-null   object
 11  Year-Month    1194 non-null   object
dtypes: int64(3), object(9)
memory usage: 112.1+ KB


### Checking for duplicates

In [3]:
df.duplicated().any()

np.False_

### Standardize column names

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ","_")
    .str.replace("-","_")
    )

### Rename specific Column Names

In [5]:
# Rename specific columns
df = df.rename(columns={
    "customername":"customer_name",
    "paymentmode":"payment_mode"
})
df.columns

Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub_category',
       'payment_mode', 'order_date', 'customer_name', 'state', 'city',
       'year_month'],
      dtype='object')

### Fix Data Types

In [6]:
# Convert order_date to datetime format
df["order_date"] = pd.to_datetime(df["order_date"])
# Convert year-month to proper datetime
df["year_month"] = pd.to_datetime(df["year_month"], format = "%Y-%m")
# Convert amount and profit to float
df["amount"] = df["amount"].astype(float)
df["profit"] = df["profit"].astype(float)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       1194 non-null   object        
 1   amount         1194 non-null   float64       
 2   profit         1194 non-null   float64       
 3   quantity       1194 non-null   int64         
 4   category       1194 non-null   object        
 5   sub_category   1194 non-null   object        
 6   payment_mode   1194 non-null   object        
 7   order_date     1194 non-null   datetime64[ns]
 8   customer_name  1194 non-null   object        
 9   state          1194 non-null   object        
 10  city           1194 non-null   object        
 11  year_month     1194 non-null   datetime64[ns]
dtypes: datetime64[ns](2), float64(2), int64(1), object(7)
memory usage: 112.1+ KB


### Create Dimension Table

In [7]:
try:
    product_dim = df[["category","sub_category"]].copy().drop_duplicates().reset_index(drop=True)
    product_dim["product_id"] = product_dim.index + 1

    customer_dim = df[["customer_name"]].copy().drop_duplicates().reset_index(drop=True)
    customer_dim["customer_id"] = customer_dim.index + 1

    payment_dim = df[["payment_mode"]].copy().drop_duplicates().reset_index(drop=True)
    payment_dim["payment_id"] = payment_dim.index + 1

    location_dim = df[["state","city"]].copy().drop_duplicates().reset_index(drop=True)
    location_dim["location_id"] = location_dim.index + 1

    date_dim = df[["order_date"]].copy().drop_duplicates().reset_index(drop=True)
    date_dim["date_id"] = date_dim.index + 1

# Extract date attributes
    date_dim["year"] = date_dim["order_date"].dt.year
    date_dim["day_name"] = date_dim["order_date"].dt.day_name()
    date_dim["month_name"] = date_dim["order_date"].dt.month_name()
    date_dim["quarter"] = date_dim["order_date"].dt.quarter
    date_dim.head()

    print("All Dimension tables created successfully")
except Exception as e:
    print("Unable to create dimension tables",e)


All Dimension tables created successfully


### Merge Dimension IDs Back to Main Data

In [8]:

df = df.merge(product_dim, on = ["category","sub_category"])
df = df.merge(customer_dim, on = "customer_name")
df = df.merge(payment_dim, on = "payment_mode")
df = df.merge(location_dim, on = ["state","city"])
df = df.merge(date_dim,on= "order_date") 

### Create Fact Table (Orders)

In [9]:
try:
    orders_fact = df[["order_id","customer_id","product_id","date_id","location_id","payment_id","amount","quantity","profit",
    ]].copy()

    orders_fact["order_items_id"] = range(1,len(orders_fact) + 1)
    orders_fact.head(3)

    print("Fact table successfully created")
except Exception as e:
    print("Unable to create the Fact table",e)

Fact table successfully created


### Export Cleaned Data

In [10]:
customer_dim.to_csv(r"Dataset\clean_dataset\customers.csv",index = False)
product_dim.to_csv(r"Dataset\clean_dataset\products.csv",index = False)
location_dim.to_csv(r"Dataset\clean_dataset\locations.csv",index = False)
payment_dim.to_csv(r"Dataset\clean_dataset\payments.csv",index = False)
orders_fact.to_csv(r"Dataset\clean_dataset\orders.csv",index = False)
date_dim.to_csv(r"Dataset\clean_dataset\dates.csv",index = False)



### viewing the columns

In [11]:
df.columns

Index(['order_id', 'amount', 'profit', 'quantity', 'category', 'sub_category',
       'payment_mode', 'order_date', 'customer_name', 'state', 'city',
       'year_month', 'product_id', 'customer_id', 'payment_id', 'location_id',
       'date_id', 'year', 'day_name', 'month_name', 'quarter'],
      dtype='object')

### Transactional tables

In [12]:
try:
    payments = df[["payment_id","payment_mode"]].copy().drop_duplicates().reset_index(drop=True)

    customers= df[["customer_id","customer_name"]].copy().drop_duplicates().reset_index(drop=True)

    products = df[["category","sub_category","amount"]].copy().drop_duplicates(subset=["sub_category"]).reset_index(drop=True)
    products['product_id']= product_dim.index + 1


    locations = df[["location_id","city","state"]].copy().drop_duplicates().reset_index(drop=True)


    orders = df[["order_id","payment_id","customer_id","product_id","location_id","order_date","quantity","profit"]].copy()
    orders["order_items_id"] = range(1,len(orders) + 1)
    orders = orders[["order_items_id","order_id","payment_id","customer_id","product_id","location_id","order_date","quantity","profit"]].copy()
    orders.head(3)

    print("Transactions table exported")
except Exception as e:
    print("Unable to create transactional tables")

Transactions table exported


### Export the Transactional tables

In [13]:
payments.to_csv(r"Dataset\transactions_data\payments.csv", index=False)
customers.to_csv(r"Dataset\transactions_data\customers.csv",index=False)
products.to_csv(r"Dataset\transactions_data\products.csv",index=False)
locations.to_csv(r"Dataset\transactions_data\locations.csv",index=False)
orders.to_csv(r"Dataset\transactions_data\orders.csv",index=False)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1194 entries, 0 to 1193
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       1194 non-null   object        
 1   amount         1194 non-null   float64       
 2   profit         1194 non-null   float64       
 3   quantity       1194 non-null   int64         
 4   category       1194 non-null   object        
 5   sub_category   1194 non-null   object        
 6   payment_mode   1194 non-null   object        
 7   order_date     1194 non-null   datetime64[ns]
 8   customer_name  1194 non-null   object        
 9   state          1194 non-null   object        
 10  city           1194 non-null   object        
 11  year_month     1194 non-null   datetime64[ns]
 12  product_id     1194 non-null   int64         
 13  customer_id    1194 non-null   int64         
 14  payment_id     1194 non-null   int64         
 15  location_id    1194 n